# 🧠 Super-Accurate Letter Recognition (EMNIST) for 3D Neural Flow

This notebook trains a high-accuracy neural network ($784 \to 64 \to 32 \to 26$) on **real human-handwritten letters** from the **EMNIST-Letters** dataset (~145,600 handwritten samples from thousands of human writers).

### Instructions:
1. Click **Runtime** > **Change runtime type** > Select **T4 GPU** (or CPU, it runs in ~20 seconds).
2. Click **Runtime** > **Run all** (or press `Ctrl + F9`).
3. At the end, `mlp_weights.json` will automatically download to your computer.
4. Replace `data/model_weights.json` and `public/exports/mlp_weights.json` in your `neural-flow` folder!

In [ ]:
# Step 1: Check GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Step 2: Download EMNIST-Letters (145,600 real handwritten letters)
import torchvision
import torchvision.transforms as transforms
from PIL import Image

# EMNIST raw images are stored transposed (swapped axes), so Image.TRANSPOSE restores them upright
transform = transforms.Compose([
    lambda img: img.transpose(Image.TRANSPOSE),
    transforms.ToTensor(),
])

print('Downloading EMNIST Letters dataset...')
train_dataset = torchvision.datasets.EMNIST(
    root='./data',
    split='letters',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.EMNIST(
    root='./data',
    split='letters',
    train=False,
    download=True,
    transform=transform
)

print('Train samples:', len(train_dataset))
print('Test samples:', len(test_dataset))


In [ ]:
# Step 3: Compute normalization and create DataLoaders
from torch.utils.data import DataLoader
import numpy as np

# EMNIST letters labels are 1-26, so subtract 1 to get 0-25 (A=0, ..., Z=25)
class ZeroIndexedEMNIST(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        return x.view(-1), y - 1

train_ds = ZeroIndexedEMNIST(train_dataset)
test_ds = ZeroIndexedEMNIST(test_dataset)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

# Normalization parameters
mean = 0.1736
std = 0.3242

In [ ]:
# Step 4: Define Neural Network (784 -> 64 -> 32 -> 26)
import torch.nn as nn
import torch.nn.functional as F

model = nn.Sequential(
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 26)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Step 5: Fast GPU Training (15 Epochs takes ~15 seconds on Colab GPU!)
import time
start_time = time.time()

for epoch in range(15):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        xb_norm = (xb - mean) / std
        
        optimizer.zero_grad()
        logits = model(xb_norm)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
        
    scheduler.step()
    train_acc = correct / total
    
    # Quick test eval
    model.eval()
    test_correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            xb_norm = (xb - mean) / std
            logits = model(xb_norm)
            test_correct += (logits.argmax(1) == yb).sum().item()
    test_acc = test_correct / len(test_ds)
    
    print(f"Epoch {epoch+1:2d}/15 | Loss: {total_loss/total:.4f} | Train Acc: {train_acc*100:.1f}% | Test Acc: {test_acc*100:.1f}%")

print(f"\nTraining complete in {time.time() - start_time:.1f}s!")

In [ ]:
# Step 6: Generate Clean Preset Letter Templates (A-Z)
from PIL import Image, ImageDraw, ImageFont

CLASSES = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
templates = []

# Fallback font rendering
try:
    fnt = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
except:
    fnt = ImageFont.load_default()

for ci, ch in enumerate(CLASSES):
    im = Image.new('L', (28, 28), 0)
    d = ImageDraw.Draw(im)
    bb = d.textbbox((0, 0), ch, font=fnt)
    tw, th = bb[2] - bb[0], bb[3] - bb[1]
    cx, cy = (28 - tw) // 2 - bb[0], (28 - th) // 2 - bb[1]
    d.text((cx, cy), ch, fill=255, font=fnt)
    arr = np.array(im, dtype=np.float32) / 255.0
    templates.append(arr.flatten().tolist())

In [ ]:
# Step 7: Export model to JSON and auto-download
import json

layers_data = [
    {
        "activation": "relu",
        "weights": model[0].weight.detach().cpu().numpy().tolist(),
        "biases": model[0].bias.detach().cpu().numpy().tolist(),
    },
    {
        "activation": "relu",
        "weights": model[2].weight.detach().cpu().numpy().tolist(),
        "biases": model[2].bias.detach().cpu().numpy().tolist(),
    },
    {
        "activation": "linear",
        "weights": model[4].weight.detach().cpu().numpy().tolist(),
        "biases": model[4].bias.detach().cpu().numpy().tolist(),
    }
]

export_dict = {
    "mode": "inference",
    "classes": CLASSES,
    "architecture": [784, 64, 32, 26],
    "normalization": {"mean": float(mean), "std": float(std)},
    "layers": layers_data,
    "templates": templates
}

output_file = "mlp_weights.json"
with open(output_file, "w") as f:
    json.dump(export_dict, f)

print(f"Successfully saved {output_file}!")

# Download file to your browser
try:
    from google.colab import files
    files.download(output_file)
    print("Triggered automatic download of mlp_weights.json!")
except ImportError:
    print("Not in Colab - file saved locally.")